In [ ]:
import getpass
from pathlib import Path
from Physical_Quantity.Physical_Data_Record import CorrelationRecord
from Physical_Quantity.Correlation_Process import CorrelationProcessor

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------
#---------------------########### A、B子格下的自旋-自旋关联函数<S_i·S_j>数据提取与作图 ################--------------------------
### 基本参数
N = 200
a = 1.0

### ================= 文件夹路径变量（需要修改的部分）=====================
δ = "d1_10"               # 空穴率
bond_dim = "1500"         # bond dimension
t_J = "t_J_3"             # t/J
version_A = "49"          # A子晶格文件夹
version_B = "50"          # B子晶格文件夹


### 配置两个子格：名称、版本号、源点偏移、最大场点偏移 
configs = [
    ("A", version_A, 1, -3),   # A子格：version=49, 源点偏移+1, 最大场点偏移-3
    ("B", version_B, 2, -2),   # B子格：version=50, 源点偏移+2, 最大场点偏移-2
]

### 保存A、B子格的关联函数对象
results = {}  

### 将输入的数据提取并封装成CorrelationRecord对象列表
for name, version, src_off, max_off in configs:
    ## 文件夹路径与文件读取
    base_dir = Path(f'Data_N{N}/{t_J}/{δ}/{bond_dim}_measure_phys_qty/measure_{version}')
    data_SuSd = (base_dir / f'sm{version}sp.json').read_text(encoding='utf-8')
    data_SdSu = (base_dir / f'sp{version}sm.json').read_text(encoding='utf-8')
    data_SzSz = (base_dir / f'sz{version}sz.json').read_text(encoding='utf-8')

    ## 生成A、B子格序列
    source_point = int(N / 8 / 4) * 8 + src_off
    field_point_max = int(N / 4 * 3) + max_off
    sequence = range(source_point, field_point_max + 1, 4)

    ## 提取数据并封装为 CorrelationRecord 对象列表
    SzSz = CorrelationRecord.extract_data(data_SzSz, sequence, N, a)
    SuSd = CorrelationRecord.extract_data(data_SdSu, sequence, N, a)  # 注意交叉对应
    SdSu = CorrelationRecord.extract_data(data_SuSd, sequence, N, a)

    ## 组合得到 <S_i·S_j>
    SiSj = CorrelationProcessor.Combined_spin_correlation(SzSz, SuSd, SdSu)

    ## 打印前5个数据
    print(f"\n----------------------------- {name}子格 自旋关联函数数据 -----------------------------------")
    for record in SiSj[:5]:
        print(record)
    print(f"... 还有{len(SiSj)-5}个点")

    ## 可视化
    CorrelationProcessor.Spin_Corr_Visualization(SiSj)

    ## 保存结果，便于后续使用
    results[name] = SiSj

### 如需分别引用：
SiSj_A_Ultra = results["A"]
SiSj_B_Ultra = results["B"]

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#---------------------########### Itensor计算A子格下的自旋-自旋关联函数<S_i·S_j>数据提取与作图 ################--------------------------
### 基本参数
N = 200  # 格点数
a = 1.0  # honeycomb晶格常数

### 配置两个子格：名称、源点偏移、最大场点偏移
configs = [
    ("A", 1, -3),   # A子格：源点偏移+1, 最大场点偏移-3
    ("B", 2, -2),   # B子格：源点偏移+2, 最大场点偏移-2
]

### 保存A、B子格的关联函数对象
results = {}  

### 将输入的数据提取并封装成CorrelationRecord对象列表
for name, src_off, max_off in configs:
    ## 输入数据
    data_SiSj = getpass.getpass(f"请输入{name}子格<S_i·S_j>: ")

    ## 生成子格序列
    source_point = int(N / 8 / 4) * 8 + src_off
    field_point_max = int(N / 4 * 3) + max_off
    sequence = range(source_point, field_point_max + 1, 4)

    ## 数据封装与打印
    SiSj = CorrelationRecord.extract_data(data_SiSj, sequence, N, a)
    print(f"\n----------------------------- {name}子格 自旋关联函数数据 -----------------------------------")
    for record in SiSj[:5]:
        print(record)
    print(f"... 还有{len(SiSj)-5}个点")

    ## 可视化
    CorrelationProcessor.Spin_Corr_Visualization(SiSj)

    ## 保存结果，方便后续分别使用
    results[name] = SiSj

### 如需分别引用变量，可这样赋值：
SiSj_A_Itensor = results["A"]
SiSj_B_Itensor = results["B"]

In [ ]:
#--------------------------------------------------------------------------------------------------------------------------
#---------------------########### UltraDMRG相对于Itensor的自旋关联函数的绝对误差 ################--------------------------

directions_A = {
    'UltraDMRG': SiSj_A_Ultra,
    'Itensor': SiSj_A_Itensor
}

directions_B = {
    'UltraDMRG': SiSj_B_Ultra,
    'Itensor': SiSj_B_Itensor
}

CorrelationProcessor.Spin_Corr_Visualization_difference(directions_A)
CorrelationProcessor.Spin_Corr_Visualization_difference(directions_B)